In [4]:
!pip install pandas numpy scikit-learn xgboost imbalanced-learn

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

In [7]:
import zipfile

with zipfile.ZipFile("/content/UCI_Credit_Card.csv.zip","r") as zf:
    zf.extractall()

data = pd.read_csv("UCI_Credit_Card.csv")

print(data.head())

   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1    20000.0    2          2         1   24      2      2     -1     -1   
1   2   120000.0    2          2         2   26     -1      2      0      0   
2   3    90000.0    2          2         2   34      0      0      0      0   
3   4    50000.0    2          2         1   37      0      0      0      0   
4   5    50000.0    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...        0.0        0.0        0.0       0.0     689.0       0.0   
1  ...     3272.0     3455.0     3261.0       0.0    1000.0    1000.0   
2  ...    14331.0    14948.0    15549.0    1518.0    1500.0    1000.0   
3  ...    28314.0    28959.0    29547.0    2000.0    2019.0    1200.0   
4  ...    20940.0    19146.0    19131.0    2000.0   36681.0   10000.0   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  default.payment.next.month  
0       0.0       0.0   

In [8]:
data = data.drop("ID", axis=1)

In [9]:
X = data.drop("default.payment.next.month", axis=1)
y = data["default.payment.next.month"]

In [10]:
scaler = StandardScaler()
X["LIMIT_BAL"] = scaler.fit_transform(X[["LIMIT_BAL"]])

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_smote.value_counts())

Before SMOTE: default.payment.next.month
0    18677
1     5323
Name: count, dtype: int64
After SMOTE: default.payment.next.month
0    18677
1    18677
Name: count, dtype: int64


In [13]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_smote, y_train_smote)

y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:,1]

print("Logistic Regression")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Logistic Regression
              precision    recall  f1-score   support

           0       0.86      0.74      0.80      4687
           1       0.39      0.59      0.47      1313

    accuracy                           0.70      6000
   macro avg       0.63      0.66      0.63      6000
weighted avg       0.76      0.70      0.72      6000

ROC-AUC: 0.7120644988626155


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
rf_model = RandomForestClassifier(n_estimators=100)

rf_model.fit(X_train_smote, y_train_smote)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:,1]

print("Random Forest")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest
              precision    recall  f1-score   support

           0       0.86      0.86      0.86      4687
           1       0.51      0.50      0.50      1313

    accuracy                           0.78      6000
   macro avg       0.68      0.68      0.68      6000
weighted avg       0.78      0.78      0.78      6000

ROC-AUC: 0.7462378723799085


In [15]:
xgb_model = XGBClassifier(eval_metric='logloss')

xgb_model.fit(X_train_smote, y_train_smote)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:,1]

print("XGBoost")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

XGBoost
              precision    recall  f1-score   support

           0       0.85      0.86      0.86      4687
           1       0.49      0.47      0.48      1313

    accuracy                           0.78      6000
   macro avg       0.67      0.67      0.67      6000
weighted avg       0.77      0.78      0.78      6000

ROC-AUC: 0.7461756367493112


In [16]:
from sklearn.neural_network import MLPClassifier

nn_model = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300)

nn_model.fit(X_train_smote, y_train_smote)

y_pred = nn_model.predict(X_test)
y_prob = nn_model.predict_proba(X_test)[:,1]

print("Neural Network")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Neural Network
              precision    recall  f1-score   support

           0       0.86      0.70      0.77      4687
           1       0.36      0.60      0.45      1313

    accuracy                           0.68      6000
   macro avg       0.61      0.65      0.61      6000
weighted avg       0.75      0.68      0.70      6000

ROC-AUC: 0.6836670955996159
